# 治したい症状に関わる遺伝子を順位づける（一続きのプログラム版）

`gene_disease_ranking02.ipynb` と同じ処理を、**上から順に読めば流れが分かる形**で書いたものです。
関数（`def`）は、何度も使う 2 つ（モデルに聞く・Bradley-Terry で強さを出す）だけにしています。
設定も YAML ではなく、このノートブックの中に書きます。

## 全体の流れ

| 段階 | やること | 既定の規模 | モデルに聞く回数 |
|---|---|---|---|
| 0 | 1 遺伝子ずつ「働きを変えると、疾患またはその症状が良くなる可能性があるか」を Yes / No で聞く | 全候補 → 2,000 | 候補数 |
| 0.5 | 5 遺伝子＋「その他」から 1 つ選ばせる。全員 4 回ずつ出題し、粗く並べる | 2,000 → 500 | 1,600 |
| 1 | 同じ選択式を全員 16 回ずつ出題し、詳しく並べる | 500 → 100 | 1,600 |
| 2 | 2 遺伝子＋「どちらも関係ない」の総当たり（A/B を入れ替えて 2 回） | 100 → 順位 | 9,900 |

qwen3:14b・19,297 遺伝子で、全体で約 5 時間 15 分でした（統合失調症の実行）。

## モデルに遺伝子記号を書かせない

どの段階も、モデルが次に出す **1 トークンの確率**だけを読みます（段階0 は `Yes` / `No`、段階0.5 以降は `A`〜`F`）。
遺伝子記号は複数のトークンに分かれるので、書かせると GPR52 と GPR56 のような似た記号を取り違えます。
記号と選択肢の文字の対応は、プログラムの側が持ちます。

## 結果を読む前に（これまでの測定で分かったこと）

- **上位の顔ぶれは、どの症状を書くかで動きます。** 統合失調症の段階0 で、陽性症状を書くと C4A 4 位・COMT 14 位、
  陰性症状・認知機能を書くと C4A 23 位・COMT 4 位でした。結果は「この疾患とこの症状に関わる遺伝子」として読んでください。
- **質問文は「疾患またはその症状が良くなる可能性があるか（potentially improve）」です。** 以前の質問文で測ったこと:
  「症状が良くなるか（improve）」では DRD2・DRD3 が上がった一方、抗精神病薬の標的 HTR2A も上位に並び、GRIN2A は選択式で下がりました。
  「疾患または症状を変える可能性があるか（potentially change）」では GRIN2A が 0.34 → 0.61 に戻りましたが、HTR2A は 0.90 のままでした。
  今の質問文では GRIN2A 0.53・COMT 0.66・DRD3 0.75 と両者の中間で、疾患そのものも対象に含められるのでこの形にしています。
  HTR2A は 0.92 のままで、薬の標的への寄りは残ります。
- **モデルが知らないつながりは拾えません。** 承認薬の標的 CHRM4 は、聞き方をどう変えても下位のままでした。
  下位にあることは「関係ない」の証拠になりません。
- 各ステップで何をしているかの説明（研究者向け）は `HOW_IT_WORKS.md` に、測定の記録は `EXPERIMENTS.md` の「ノートブック 02」の節にあります。

## 1. 入力 — 疾患・治したい症状・候補遺伝子

**解析ごとに変えるのはこのセルだけです。** 用意した疾患パターンから `SELECTED` で 1 つ選びます。
自分で書くときは `SELECTED = "custom"` にして、`CUSTOM` を埋めます。

モデルには、疾患名と「**治したい症状（部位）**」の一覧を見せ、「この遺伝子の働きを変えると、疾患またはこれらの症状が良くなる可能性があるか」を聞きます。
病気の仕組み（メカニズム）は書きません。

| `SELECTED` | 疾患 | 治したい症状 | 候補遺伝子 |
|---|---|---|---|
| `"cystinuria"` | シスチン尿症 | 繰り返す腎結石・結石の痛み・腎機能の低下 | 1,000（`genelist_cystinuria_mech1000.txt`） |
| `"cah"` | 先天性副腎過形成症 | 塩類喪失・女児の外性器の男性化・早い思春期・多毛 | 1,000（`genelist_cah_mech1000.txt`） |
| `"schizophrenia_positive"` | 統合失調症 | 陽性症状（幻聴・妄想・まとまらない思考） | 1,000（`genelist_schizophrenia_mech1000.txt`） |
| `"schizophrenia_negative"` | 統合失調症 | 陰性症状・認知機能（意欲低下・感情の平板化・作業記憶） | 同上 |
| `"type2_diabetes"` | 2 型糖尿病 | 高血糖・口渇と多尿・目や腎臓や神経の傷み | 19,297（`genelist01_all.txt`） |
| `"achondroplasia"` | 軟骨無形成症 | 手足が短い・低身長・頭蓋底や腰の脊柱管の狭さ | 19,297（`genelist01_all.txt`） |

統合失調症の 2 つは、**同じ疾患で治したい症状が違う**だけです。上位の顔ぶれは症状の選び方で動きます
（qwen3:14b・1,000 遺伝子の段階0 で、C4A は陽性症状 4 位 → 陰性症状 23 位、COMT は 14 位 → 4 位）。
大事な判断には両方で回して、どちらでも上位に来る遺伝子を見てください。
候補数を変えたいときは、パターンの `gene_file` を書き換えます（全遺伝子は `genelist01_all.txt`、約 5 時間）。

**症状の書き方**: 1 行に 1 つ、患者さんに現れる症状を書き、最後に部位を括弧で添えます。
例: `"Hallucinations, mostly hearing voices that others do not hear (brain)"`。
遺伝子名・受容体名・薬の名前・病気の仕組みは書きません（答えを教えることになる）。

**この形で分かっていること**（統合失調症、`EXPERIMENTS.md`）: 前の「メカニズム」の形に比べて、ドパミン受容体が上がります
（選択式で DRD2 0.27 → 0.85、DRD3 0.20 → 0.67）。代わりに、症状を直接治す薬の標的（HTR2A など）も一緒に上がり、
遺伝学で見つかった GRIN2A は選択式で下がりました（0.99 → 0.34）。これは質問文が「症状が良くなるか（improve）」のときの測定です。
「疾患または症状を変える可能性があるか（potentially change）」では GRIN2A が 0.61 に戻り、COMT も 0.50 → 0.70 に上がりましたが、
HTR2A は 0.90 のままでした。今の「疾患またはその症状が良くなる可能性があるか（potentially improve）」は、その中間（GRIN2A 0.53、COMT 0.66、HTR2A 0.92）でした。

候補遺伝子のファイルは、1 行 1 遺伝子のタブ区切りです:
`記号<TAB>HGNC 正式名<TAB>UniProt の蛋白質名<TAB>別名（カンマ区切り）`（2〜4 列目は無くても動きます）。
`known_answers` は答え合わせ用で、空でも構いません。根拠は各パターンの行のコメントにあります。

In [ ]:
SELECTED = "schizophrenia_positive"      # 下の DISEASE_PATTERNS の名前か "custom"

DISEASE_PATTERNS = {
    "cystinuria": dict(
        disease="cystinuria",
        symptoms=[
            "Kidney stones that keep coming back (kidneys, urinary tract)",
            "Sudden severe pain in the side or lower back when a stone passes (kidneys, ureter)",
            "Blood in the urine and urinary tract infections (urinary tract)",
            "Gradual loss of kidney function after repeated stones and blockage (kidneys)",
        ],
        gene_file="genelist_cystinuria_mech1000.txt",
        # SLC3A1・SLC7A9: 原因（Open Targets 0.85）。SLC5A2: dapagliflozin NCT04818034。AVPR2: tolvaptan NCT02538016
        known_answers=["SLC3A1", "SLC7A9", "SLC5A2", "AVPR2"],
    ),
    "cah": dict(
        disease="congenital adrenal hyperplasia",
        symptoms=[
            "Salt-wasting crises in newborns, with vomiting, dehydration and low blood pressure (kidneys, circulation)",
            "Masculinized genitals in newborn girls (external genitals)",
            "Early puberty and fast growth in childhood that ends in short adult height (whole body, bones)",
            "Excess facial and body hair, acne and irregular periods in women (skin, ovaries)",
        ],
        gene_file="genelist_cah_mech1000.txt",
        # CYP21A2: 原因（Open Targets 0.853）。CRHR1: crinecerfont（ChEMBL: CRF1 拮抗薬）。MC2R: atumelnant NCT05907291
        known_answers=["CYP21A2", "CRHR1", "MC2R"],
    ),
    "schizophrenia_positive": dict(
        disease="schizophrenia",
        symptoms=[
            "Hallucinations, mostly hearing voices that others do not hear (brain)",
            "Delusions, fixed false beliefs such as being watched or persecuted (brain)",
            "Disorganized thinking and speech (brain)",
        ],
        gene_file="genelist_schizophrenia_mech1000.txt",
        # DRD2・DRD3: 抗精神病薬（Open Targets 臨床 1.00）。CHRM4: xanomeline（M4/M1 作動薬）。GRIN2A・GRM3: 遺伝学。C4A: 文献
        known_answers=["C4A", "GRIN2A", "GRM3", "DRD2", "DRD3", "CHRM4", "COMT", "GAD1", "SLC6A3"],
    ),
    "schizophrenia_negative": dict(
        disease="schizophrenia",
        symptoms=[
            "Loss of motivation and withdrawal from other people (brain)",
            "Flat emotional expression and reduced speech (brain)",
            "Poor working memory and attention (brain)",
        ],
        gene_file="genelist_schizophrenia_mech1000.txt",
        known_answers=["C4A", "GRIN2A", "GRM3", "DRD2", "DRD3", "CHRM4", "COMT", "GAD1", "SLC6A3"],
    ),
    "type2_diabetes": dict(
        disease="type 2 diabetes",
        symptoms=[
            "High blood glucose after meals and on waking (blood)",
            "Excessive thirst and frequent urination (kidneys)",
            "Damage to the small blood vessels of the eyes and kidneys over the years (eyes, kidneys)",
            "Numbness and pain in the feet (peripheral nerves)",
        ],
        gene_file="genelist01_all.txt",
        # TCF7L2・KCNJ11・PPARG: 遺伝学（Open Targets 0.77〜0.93）。PPARG: pioglitazone。GLP1R: semaglutide。SLC5A2: dapagliflozin
        known_answers=["TCF7L2", "KCNJ11", "PPARG", "GLP1R", "SLC5A2"],
    ),
    "achondroplasia": dict(
        disease="achondroplasia",
        symptoms=[
            "Short arms and legs with a normal-sized trunk (long bones)",
            "Short adult height (whole skeleton)",
            "A narrow opening at the base of the skull that presses on the spinal cord in infants (skull base, spinal cord)",
            "A narrow spinal canal in the lower back that causes leg pain and weakness in adults (lumbar spine)",
        ],
        gene_file="genelist01_all.txt",
        # FGFR3: 原因（Open Targets 0.824）。NPR2: vosoritide（ChEMBL、承認）。NPPC: TransCon CNP NCT05598320
        known_answers=["FGFR3", "NPR2", "NPPC"],
    ),
}

# 自分で書くとき（SELECTED = "custom"）
CUSTOM = dict(
    disease="",
    symptoms=[],                   # 治したい症状を 1 行に 1 つ、最後に部位を括弧で
    gene_file="genelist01_all.txt",
    known_answers=[],
)

# ----- 選んだパターンを取り出す -----
if SELECTED == "custom":
    pattern = CUSTOM
elif SELECTED in DISEASE_PATTERNS:
    pattern = DISEASE_PATTERNS[SELECTED]
else:
    raise ValueError(f"SELECTED = {SELECTED!r} はありません。選べるのは {list(DISEASE_PATTERNS)} か 'custom' です")
if not pattern["disease"] or not pattern["symptoms"]:
    raise ValueError("disease と symptoms（1 つ以上）を書いてください")

DISEASE = pattern["disease"]
DISEASE_SYMPTOMS = list(pattern["symptoms"])
GENE_FILE = pattern["gene_file"]
KNOWN_ANSWERS = list(pattern["known_answers"])

print(f"パターン: {SELECTED}  疾患: {DISEASE}  候補: {GENE_FILE}")
print("治したい症状:")
for symptom in DISEASE_SYMPTOMS:
    print(f"  - {symptom}")
print("答え合わせ:", KNOWN_ANSWERS)

## 2. 設定

ふだんは変えなくて構いません。数字はこれまでの測定で決めたものです。

- **モデル**: `qwen3:14b`。このマシン（VRAM 17.8 GiB）では、別の大きなモデル（gemma3:27b など）と同時には載りません。
  同時に使うと、呼び出しのたびに載せ替えが起きて数十倍遅くなります。
- **モデルの取り込み（`MODEL_IMPORT`）**: Ollama にまだ無いモデル（meditron-70b など）を使うときに、取り込み元を書きます。
  `MODEL` の名前で Ollama に登録し、次からはそのまま使えます。Hugging Face の元の重み（`epfl-llm/meditron-70b`）、
  量子化済みの GGUF、手元のファイルのどれからでも取り込めます（書き方と確認の順番は 4. にあります）。

  **大きさに注意してください。** meditron-70b の元の重みは 138 GB（16 ビット）で、Q4_K_M に量子化して約 41 GB になります。
  このマシン（Apple M4・メモリ 25.8 GB・空きディスク 165 GB）ではメモリもディスクも足りないので、
  メモリ 48 GB 以上・空きディスク 320 GB 以上のサーバーで取り込み、`OLLAMA_HOST` でそのサーバーを指します。
  meditron は会話用ではない素のモデル（Llama 2 系、文脈 4,096 トークン）ですが、このノートブックは例題を並べて
  続きの 1 文字を読む形なので、そのまま使えます。ただしこの課題での精度は未測定です
  （医学特化の biomistral 7B は、選択式で 1 文字を返さなかったことがあります）。
- **残す数と登場回数**: 段階0.5 は「粗く並べて真の上位を落とさない」役目なので 4 回で足ります
  （シミュレーションで、真の上位 20 の 100% が 500 個に残った）。段階1 は 16 回で、真の上位 20 が 100 個に残りました。
- **質問文**: 段階0 と選択式で文の形が違います（Yes/No で答える形と、どれかを選ぶ形）。
- **見本（few-shot）**: 本番と同じ形の例題を先に見せます。題材は嚢胞性線維症と関節リウマチで、
  対象の疾患とは別にしてあります。段階0 の見本で DNASE1（原因ではないが、粘液を薄くする）を Yes にしているのは、
  「原因遺伝子だけが Yes」ではないことを示すためです。

In [ ]:
# ===== モデルと Ollama =====
MODEL = "qwen3:14b"
MODEL_IMPORT = ""         # MODEL が Ollama に無いときの取り込み元。空なら取り込まない（書き方は 4. の表）。例:
                          #   MODEL = "meditron-70b"
                          #   MODEL_IMPORT = "epfl-llm/meditron-70b"                     （元の重み 138 GB をダウンロードして量子化）
                          #   MODEL_IMPORT = "hf.co/TheBloke/meditron-70B-GGUF:Q4_K_M"   （量子化済みの GGUF 41.4 GB）
                          #   MODEL_IMPORT = "/data/models/meditron-70b"                 （手元の safetensors のフォルダ）
MODEL_QUANTIZE = "q4_K_M" # safetensors を取り込むときの量子化（"q4_K_M" / "q4_K_S" / "q8_0" / "" は 16 ビットのまま）
MODEL_DOWNLOAD_DIR = ""   # safetensors のダウンロード先。空なら Hugging Face の既定（~/.cache/huggingface）
OLLAMA_HOST = ""          # 空なら環境変数 OLLAMA_HOST、それも無ければ localhost:11434
TIMEOUT_SEC = 180

# ===== 各段階で残す数・1 遺伝子あたりの登場回数 =====
STAGE0_KEEP = 2000        # 段階0 から段階0.5 へ送る数（候補数より多ければ全員送る）
GROUP_SIZE = 5            # 選択式の 1 問に並べる遺伝子の数（「その他」を除く）
CHOICE_ROUNDS = [         # (名前, 残す数, 1 遺伝子の登場回数, 乱数の種)
    ("段階0.5", 500, 4, 0),
    ("段階1", 100, 16, 1),
]
BOTH_WAYS = True          # 段階2 で A/B を入れ替えて 2 回聞く（選択肢の順番の偏りを打ち消す）

# ===== 質問文 =====
# 「良くなるか（improve）」と聞くと、今ある薬の標的に寄った（統合失調症で HTR2A 0.92、GRIN2A 0.99 → 0.34）。
# 「疾患または症状を変える可能性（potentially change）」では GRIN2A 0.61・COMT 0.70 に戻った（HTR2A は 0.90 のまま）。
# 「症状が良くなるか（improve）」では GRIN2A 0.34 まで下がり、薬の標的に寄った（HTR2A 0.92）。
# 今の「疾患またはその症状が良くなる可能性（potentially improve）」は GRIN2A 0.53・COMT 0.66・DRD3 0.75 と中間で、
# 段階0 の Yes 寄りも 246 個（improve 131、change 370）と中間。疾患そのものも対象に含められるのでこの形で確定
BINARY_QUESTION = "Could changing this gene's activity, in either direction, potentially improve the disease or any of these symptoms of the disease?"
CHOICE_QUESTION = "Changing which gene's activity, in either direction, could potentially improve the disease or any of these symptoms of the disease?"
CHOICE_EXIT = "Other / none of the above"   # 段階0.5・1 の「その他」
PAIR_EXIT = "Neither of these genes"        # 段階2 の第3の選択肢

# ===== 選択肢の表示（段階0.5 以降）=====
# 記号 (蛋白質名) (aliases: 別名) の形。段階0 は記号だけ（蛋白質名を足すと正解の順位がかえって下がった）
MAX_ALIASES = 4

# ===== Bradley-Terry =====
ALPHA = 0.5               # 仮想の引き分け。一度も勝てない遺伝子の強さが -∞ に飛ぶのを防ぐ
NONE = "(none of these)"  # 「その他」「どちらも関係ない」を表す仮想の対戦相手

# ===== 見本（few-shot）=====
FEWSHOT_SYMPTOMS = {     # 見本の疾患の「治したい症状」。本番と同じ書き方にする
    "Cystic fibrosis": [
        "Thick, sticky mucus that clogs the airways (lungs)",
        "Repeated lung infections (lungs)",
    ],
    "Rheumatoid arthritis": [
        "Painful, swollen and stiff joints (hands, feet and other joints)",
        "Gradual destruction of cartilage and bone in the joints (joints)",
    ],
}
BINARY_EXAMPLES = [("CFTR", "Yes"), ("APOE", "No"), ("DNASE1", "Yes"), ("HBB", "No")]   # 段階0（嚢胞性線維症）
CHOICE_EXAMPLES = [   # 段階0.5・1。正解の文字を B と C に分ける（同じ文字に寄せると、その文字を選ぶ癖がつく）
    ("Cystic fibrosis", ["HBB", "CFTR", "GPR52", "APOE"], "CFTR"),
    ("Rheumatoid arthritis", ["CFTR", "HBB", "TNF", "GPR56"], "TNF"),
]
PAIR_EXAMPLES = [     # 段階2。正解は B と A に分ける
    ("Cystic fibrosis", ["HBB", "CFTR"], "CFTR"),
    ("Rheumatoid arthritis", ["TNF", "GPR56"], "TNF"),
]
FEWSHOT_PROTEIN = {
    "HBB": "Hemoglobin subunit beta", "CFTR": "Cystic fibrosis transmembrane conductance regulator",
    "GPR52": "G-protein coupled receptor 52", "APOE": "Apolipoprotein E", "TNF": "Tumor necrosis factor",
    "GPR56": "Adhesion G-protein coupled receptor G1", "DNASE1": "Deoxyribonuclease-1",
}
FEWSHOT_ALIASES = {"CFTR": ["ABCC7"], "TNF": ["TNFA", "TNFSF2"], "GPR56": ["ADGRG1", "TM7LN4", "TM7XN1"], "DNASE1": ["DNL1", "DRNI"]}

# ===== 保存先 =====
OUTPUT_ROOT = "outputs"
RESUME_DIR = ""           # 途中から再開するときに、前回の実行フォルダ（例 "outputs/20260919-101500-nb03"）を入れる

## 3. 準備 — Ollama に繋がるか

Ollama の住所を決め、入っているモデルの一覧を取ります。
`0.0.0.0:11434` や `ollama:11434` のような、Docker でよく見るスキーム無しの書き方も読めるようにします。

In [ ]:
import hashlib, json, math, os, random, re, shutil, statistics, subprocess, time, urllib.error, urllib.request
from collections import defaultdict

host = (OLLAMA_HOST or os.environ.get("OLLAMA_HOST") or "localhost:11434").strip().rstrip("/")
if "://" not in host:
    host = "http://" + host
    if ":" not in host.split("://", 1)[1]:
        host += ":11434"
OLLAMA_URL = host.replace("0.0.0.0", "localhost")
OLLAMA_IS_LOCAL = any(h in OLLAMA_URL for h in ("localhost", "127.0.0.1"))
MODEL_TAGGED = MODEL if ":" in MODEL.rsplit("/", 1)[-1] else MODEL + ":latest"   # Ollama の一覧はタグ付きの名前で出る

with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=30) as r:
    installed = {m["name"]: m["size"] for m in json.loads(r.read())["models"]}
print("Ollama:", OLLAMA_URL, "（このマシン）" if OLLAMA_IS_LOCAL else "（別のマシン）")
print("モデル:", MODEL, f"（入っている、{installed[MODEL_TAGGED]/1e9:.1f} GB）" if MODEL_TAGGED in installed else "（まだ入っていない）")

## 4. モデルの取り込み（`MODEL` が Ollama に無いときだけ）

`MODEL_IMPORT` の書き方で、取り込み元を見分けます。

| 書き方 | 取り込み元 | やること |
|---|---|---|
| `"hf.co/<リポジトリ>:<量子化>"` | Hugging Face の GGUF | Ollama にダウンロードさせる（`ollama pull` と同じ） |
| `"/path/to/model.gguf"` | 手元の GGUF | Ollama に送って登録する |
| `"<所有者>/<名前>"`（例 `"epfl-llm/meditron-70b"`） | Hugging Face の元の重み（safetensors） | ダウンロード → Ollama に送る → `MODEL_QUANTIZE` で量子化して登録 |
| `"/path/to/folder"` | 手元の safetensors のフォルダ | Ollama に送る → 量子化して登録 |

**ダウンロードの前に、次の順で確かめ、足りなければその場で止めます。**

1. **大きさ**: Hugging Face は目録だけを取って大きさを調べます（ファイル本体はまだ落としません）。
   safetensors は量子化後の大きさも見積もります（Q4_K_M なら元の約 30%。meditron-70b は 138 GB → 約 41 GB）。
2. **メモリ**（Ollama がこのマシンのとき）: 量子化後のモデルがメモリの 9 割を超えるなら止めます。載らないモデルは、動いても極端に遅くなります。
3. **ディスク**: safetensors は、ダウンロード分に加えて、Ollama が自分の置き場に同じ中身をもう一組と量子化後のモデルを作ります。
   meditron-70b なら 138 + 138 + 41 ≒ **320 GB** の空きが要ります（取り込み後、ダウンロードしたフォルダは消して構いません）。
4. **認証**: meditron-70b は利用条件への同意が必要なリポジトリです。先に Hugging Face のモデルのページで同意し、
   読み取り用のトークンを作って、環境変数 `HF_TOKEN` に入れるか `huggingface-cli login` でログインしておきます。
   トークンはノートブックに書かないでください。
5. **ライブラリ**: safetensors のダウンロードには `huggingface_hub` を使います（無ければ `%pip install huggingface_hub`）。

送るファイルは中身の指紋（SHA-256）で見分け、Ollama に送り済みのものは送り直しません。途中で止まっても、やり直せば続きから進みます。

In [ ]:
QUANT_RATIO = {"q4_K_M": 0.30, "q4_K_S": 0.285, "q8_0": 0.53, "": 1.0}    # 量子化後 ÷ 元（16 ビット）の大きさ

if MODEL_TAGGED in installed or not MODEL_IMPORT:
    print("取り込みは不要です" if MODEL_TAGGED in installed else "MODEL_IMPORT が空なので取り込みません")
else:
    # ===== 1. 取り込み元の種類と大きさ =====
    if MODEL_IMPORT.startswith(("hf.co/", "huggingface.co/")):
        kind = "Hugging Face の GGUF"
    elif os.path.isfile(MODEL_IMPORT) and open(MODEL_IMPORT, "rb").read(4) == b"GGUF":   # GGUF はファイルの先頭 4 バイトで見分ける
        kind = "手元の GGUF"
    elif os.path.isdir(MODEL_IMPORT):
        kind = "手元の safetensors"
    elif re.fullmatch(r"[\w.-]+/[\w.-]+", MODEL_IMPORT):
        kind = "Hugging Face の safetensors"
    else:
        raise ValueError(f"MODEL_IMPORT の書き方が分かりません: {MODEL_IMPORT}")

    if kind.startswith("Hugging Face"):             # Hugging Face に問い合わせる（目録だけ。本体は落とさない）
        if kind == "Hugging Face の GGUF":
            repo, _, tag = MODEL_IMPORT.split("/", 1)[1].partition(":")
            url, accept = f"https://huggingface.co/v2/{repo}/manifests/{tag or 'latest'}", "application/vnd.docker.distribution.manifest.v2+json"
        else:
            url, accept = f"https://huggingface.co/api/models/{MODEL_IMPORT}?blobs=true", "application/json"
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"Accept": accept}), timeout=60) as r:
                info = json.loads(r.read())
        except urllib.error.HTTPError as e:
            raise RuntimeError(f"{MODEL_IMPORT} が見つかりません（HTTP {e.code}）。名前を確かめてください") from None
        except urllib.error.URLError:
            # python.org 版の Python は証明書が入っていないことがある。そのときはシステムの curl で取る
            result = subprocess.run(["curl", "-sfL", "--max-time", "60", "-H", f"Accept: {accept}", url], capture_output=True)
            if result.returncode:
                raise RuntimeError(f"{MODEL_IMPORT} が見つかりません（curl {result.returncode}）。名前を確かめてください") from None
            info = json.loads(result.stdout)

    if kind == "Hugging Face の GGUF":
        download_gb = model_gb = sum(layer["size"] for layer in info["layers"]) / 1e9
        send_gb = 0
    elif kind == "手元の GGUF":
        download_gb, send_gb = 0, os.path.getsize(MODEL_IMPORT) / 1e9
        model_gb = send_gb
    elif kind == "Hugging Face の safetensors":
        # 使うのは safetensors の重み・設定（*.json）・トークナイザだけ。同じ中身の .bin は落とさない
        wanted = [s for s in info["siblings"]
                  if s["rfilename"].endswith((".safetensors", ".json")) or s["rfilename"] == "tokenizer.model"]
        download_gb = send_gb = sum(s.get("size", 0) for s in wanted) / 1e9
        weights_gb = sum(s.get("size", 0) for s in wanted if s["rfilename"].endswith(".safetensors")) / 1e9
        model_gb = weights_gb * QUANT_RATIO[MODEL_QUANTIZE]
        gated = info.get("gated")
    else:                                              # 手元の safetensors
        names = [n for n in os.listdir(MODEL_IMPORT) if n.endswith((".safetensors", ".json")) or n == "tokenizer.model"]
        download_gb = 0
        send_gb = sum(os.path.getsize(os.path.join(MODEL_IMPORT, n)) for n in names) / 1e9
        model_gb = sum(os.path.getsize(os.path.join(MODEL_IMPORT, n)) for n in names if n.endswith(".safetensors")) / 1e9 * QUANT_RATIO[MODEL_QUANTIZE]

    print(f"取り込み: {MODEL_IMPORT}（{kind}）→ {MODEL}")
    print(f"  ダウンロード {download_gb:.1f} GB / Ollama に送る {send_gb:.1f} GB / 登録後のモデル 約 {model_gb:.1f} GB"
          + (f"（{MODEL_QUANTIZE} に量子化）" if "safetensors" in kind and MODEL_QUANTIZE else ""))

    # ===== 2. メモリ（Ollama がこのマシンのとき）=====
    if OLLAMA_IS_LOCAL:
        memory_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
        print(f"  このマシンのメモリ {memory_gb:.1f} GB")
        if model_gb > memory_gb * 0.9:
            raise MemoryError(f"約 {model_gb:.1f} GB のモデルは、このマシンのメモリ {memory_gb:.1f} GB に載りません。"
                              "小さい量子化にするか、メモリの大きいサーバーの Ollama を OLLAMA_HOST に指定してください")
    else:
        print("  （Ollama が別のマシンなので、メモリとその置き場の空きは確かめていません）")

    # ===== 3. ディスク =====
    download_dir = MODEL_DOWNLOAD_DIR or os.path.expanduser("~/.cache/huggingface")
    ollama_dir = os.environ.get("OLLAMA_MODELS") or os.path.expanduser("~/.ollama/models")
    need = defaultdict(float)                          # ディスク（装置番号）ごとに必要な量を足す
    if download_gb and "safetensors" in kind:
        os.makedirs(download_dir, exist_ok=True)
        need[os.stat(download_dir).st_dev] += download_gb
    if OLLAMA_IS_LOCAL:
        os.makedirs(ollama_dir, exist_ok=True)
        need[os.stat(ollama_dir).st_dev] += (send_gb if send_gb else download_gb) + (model_gb if "safetensors" in kind else 0)
    for device, gb in need.items():
        where = download_dir if download_gb and os.stat(download_dir).st_dev == device else ollama_dir
        free_gb = shutil.disk_usage(where).free / 1e9
        print(f"  ディスク（{where} のある所）: 必要 {gb:.0f} GB / 空き {free_gb:.0f} GB")
        if gb > free_gb:
            raise OSError(f"ディスクが足りません（必要 {gb:.0f} GB、空き {free_gb:.0f} GB）。"
                          "MODEL_DOWNLOAD_DIR を空きの大きいディスクにするか、別のサーバーで取り込んでください")

    # ===== 4. 取り込む =====
    files = {}                                         # Ollama に登録するファイル {名前: 中身の指紋}
    if kind == "Hugging Face の GGUF":
        request = urllib.request.Request(OLLAMA_URL + "/api/pull", json.dumps({"model": MODEL_IMPORT, "stream": True}).encode(),
                                         {"Content-Type": "application/json"})
        shown = -1
        with urllib.request.urlopen(request, timeout=None) as r:
            for line in r:                             # 進み具合が 1 行ずつ届く
                status = json.loads(line)
                if "error" in status:
                    raise RuntimeError(status["error"])
                if status.get("total") and status.get("completed") is not None:
                    percent = int(100 * status["completed"] / status["total"])
                    if percent // 10 > shown:
                        shown = percent // 10
                        print(f"  ダウンロード {percent}%（{status['completed']/1e9:.1f} / {status['total']/1e9:.1f} GB）", flush=True)
        # MODEL の名前でも呼べるようにする（中身は共有するので容量は増えない）
        urllib.request.urlopen(urllib.request.Request(OLLAMA_URL + "/api/copy",
            json.dumps({"source": MODEL_IMPORT, "destination": MODEL}).encode(), {"Content-Type": "application/json"}), timeout=60).close()
    else:
        # --- 送るファイルの一覧（名前, 手元の場所）---
        if kind == "手元の GGUF":
            local = [(os.path.basename(MODEL_IMPORT), MODEL_IMPORT)]
        else:
            if kind == "Hugging Face の safetensors":
                try:
                    import huggingface_hub
                except ImportError:
                    raise ImportError("huggingface_hub がありません。このノートブックで %pip install huggingface_hub を実行してください") from None
                token = os.environ.get("HF_TOKEN") or huggingface_hub.get_token()
                if gated and not token:
                    raise PermissionError(f"{MODEL_IMPORT} は利用条件への同意が必要です。https://huggingface.co/{MODEL_IMPORT} で同意し、"
                                          "読み取り用トークンを環境変数 HF_TOKEN に入れるか huggingface-cli login でログインしてください")
                print(f"  ダウンロード中（{download_gb:.0f} GB。止まっても、やり直せば続きから）…", flush=True)
                folder = huggingface_hub.snapshot_download(MODEL_IMPORT, token=token, local_dir=MODEL_DOWNLOAD_DIR or None,
                                                           allow_patterns=["*.safetensors", "*.json", "tokenizer.model"])
                print("  ダウンロード先:", folder)
            else:
                folder = MODEL_IMPORT
            local = [(n, os.path.join(folder, n)) for n in sorted(os.listdir(folder))
                     if n.endswith((".safetensors", ".json")) or n == "tokenizer.model"]

        # --- 1 ファイルずつ、指紋を取って Ollama に送る（送り済みなら飛ばす）---
        for n, (name, path) in enumerate(local, 1):
            sha = hashlib.sha256()
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(1 << 24), b""):
                    sha.update(chunk)
            digest = "sha256:" + sha.hexdigest()
            files[name] = digest
            try:                                       # 送り済みなら 200、無ければ 404
                urllib.request.urlopen(urllib.request.Request(OLLAMA_URL + f"/api/blobs/{digest}", method="HEAD"), timeout=60).close()
                note = "送り済み"
            except urllib.error.HTTPError as e:
                if e.code != 404:
                    raise
                with open(path, "rb") as f:
                    urllib.request.urlopen(urllib.request.Request(OLLAMA_URL + f"/api/blobs/{digest}", data=f, method="POST",
                                           headers={"Content-Length": str(os.path.getsize(path))}), timeout=None).close()
                note = "送った"
            print(f"  {n}/{len(local)} {name}（{os.path.getsize(path)/1e9:.2f} GB）{note}", flush=True)

        # --- 登録（safetensors はここで量子化する）---
        body = {"model": MODEL, "files": files, "stream": True}
        if "safetensors" in kind and MODEL_QUANTIZE:
            body["quantize"] = MODEL_QUANTIZE
        last = None
        with urllib.request.urlopen(urllib.request.Request(OLLAMA_URL + "/api/create", json.dumps(body).encode(),
                                                           {"Content-Type": "application/json"}), timeout=None) as r:
            for line in r:
                status = json.loads(line)
                if "error" in status:
                    raise RuntimeError(status["error"])
                if status.get("status") and status["status"] != last:      # 同じ表示は繰り返さない
                    last = status["status"]
                    print(" ", last, flush=True)
        if kind == "Hugging Face の safetensors":
            print(f"  取り込み済みです。ダウンロードしたフォルダ {folder} は消して構いません（{download_gb:.0f} GB）")

    with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=30) as r:
        installed = {m["name"]: m["size"] for m in json.loads(r.read())["models"]}
    print("モデル:", MODEL, f"（入った、{installed[MODEL_TAGGED]/1e9:.1f} GB）" if MODEL_TAGGED in installed else "⚠ 登録できませんでした")

## 5. GPU の状態と保存先

- **いま GPU に載っているモデル**を表示します。`MODEL` 以外が載っていたら、`ollama stop <名前>` で降ろしてから始めてください。
- 実行ごとに `outputs/日付-時刻-nb03/` を作り、段階ごとの結果をそこに保存します。
  `RESUME_DIR` を指定すると、そのフォルダにある段階は読み込むだけでモデルを呼びません。

In [ ]:
if MODEL_TAGGED not in installed:
    raise RuntimeError(f"{MODEL} が Ollama に入っていません。ollama pull するか、MODEL_IMPORT に取り込み元を書いてください")

with urllib.request.urlopen(OLLAMA_URL + "/api/ps", timeout=30) as r:
    resident = [m["name"] for m in json.loads(r.read())["models"]]
print("いま載っているモデル:", resident or "なし")
if [m for m in resident if m != MODEL_TAGGED]:
    print("⚠ ほかのモデルが載っています。載せ替えで遅くなるので、ollama stop で降ろしてから始めてください。")

if RESUME_DIR:
    RUN_DIR = RESUME_DIR
    print("再開:", RUN_DIR, sorted(os.listdir(RUN_DIR)))
else:
    RUN_DIR = os.path.join(OUTPUT_ROOT, time.strftime("%Y%m%d-%H%M%S") + "-nb03")
    suffix = 2
    while os.path.exists(RUN_DIR):                       # 同じ秒に 2 回始めたとき
        RUN_DIR = os.path.join(OUTPUT_ROOT, time.strftime("%Y%m%d-%H%M%S") + f"-nb03-{suffix}")
        suffix += 1
    os.makedirs(RUN_DIR)
    print("保存先:", RUN_DIR)
with open(os.path.join(RUN_DIR, "input.json"), "w") as f:     # この実行の入力を残す
    json.dump({"disease": DISEASE, "symptoms": DISEASE_SYMPTOMS, "gene_file": GENE_FILE, "model": MODEL,
               "model_import": MODEL_IMPORT, "model_quantize": MODEL_QUANTIZE,
               "stage0_keep": STAGE0_KEEP, "rounds": CHOICE_ROUNDS}, f, ensure_ascii=False, indent=1)

## 6. 候補遺伝子を読む

ファイルから、遺伝子記号の並び（`GENES`）と、選択式で見せる表示（`OPTION_TEXT`）を作ります。

表示は `記号 (蛋白質名) (aliases: 別名)` です。蛋白質名は、同じ系統の遺伝子（SLC7A9 と SLC7A11 など）を
見分けるのに効きました。別名は、よく知られた呼び名（SLC5A2 = SGLT2 など）で取り違えないために付けます。
ただし別名のうち 590 個は別の遺伝子の正式記号と同じ文字列なので、紛らわしいことがあります（AOC1 の別名 DAO など）。

In [ ]:
GENES, PROTEIN, ALIASES, seen = [], {}, {}, set()
with open(GENE_FILE) as f:
    for line in f:
        if not line.strip() or line.startswith("#"):
            continue
        cols = (line.rstrip("\n").split("\t") + ["", "", ""])[:4]
        symbol = cols[0].strip()
        if not symbol or symbol in seen:
            continue                                   # 重複は 1 回だけ数える
        seen.add(symbol)
        GENES.append(symbol)
        if cols[2].strip():
            PROTEIN[symbol] = cols[2].strip()
        if cols[3].strip():
            ALIASES[symbol] = [a.strip() for a in cols[3].split(",") if a.strip()]

# 選択式で見せる 1 行分の表示を、全員分まとめて作っておく（見本の遺伝子も含める）
OPTION_TEXT = {}
for symbol in GENES + list(FEWSHOT_PROTEIN):
    protein = PROTEIN.get(symbol) or FEWSHOT_PROTEIN.get(symbol)
    aliases = (ALIASES.get(symbol) or FEWSHOT_ALIASES.get(symbol) or [])[:MAX_ALIASES]
    text = symbol + (f" ({protein})" if protein else "")
    if aliases:
        text += f" (aliases: {', '.join(aliases)})"
    OPTION_TEXT[symbol] = text

print(f"候補 {len(GENES)} 遺伝子（{GENE_FILE}）  蛋白質名あり {len(PROTEIN)}  別名あり {len(ALIASES)}")
print("例:", OPTION_TEXT[GENES[0]])
missing = [g for g in KNOWN_ANSWERS if g not in GENES]
print("答え合わせ:", KNOWN_ANSWERS, f"⚠ 候補に無い: {missing}" if missing else "（すべて候補に含まれる）")

## 7. 道具 1 — モデルに聞いて、次の 1 トークンの確率を読む

どの段階でも使うので、ここだけ関数にします。

Ollama の `/api/generate` に、`num_predict: 1`（1 トークンだけ出す）・`temperature: 0`・`logprobs: true` で問い合わせ、
**出てきたトークンと、その次点 20 個の log 確率**を `{トークン: log 確率}` の形で返します。
`think: false` は、qwen3 が答えの前に考え始めないようにする指定です（考えると 1 トークン目が答えにならない）。

温度 0 なので、同じプロンプトには毎回同じ答えが返ります。

In [ ]:
def first_token_logprobs(prompt):
    body = json.dumps({
        "model": MODEL, "prompt": prompt, "stream": False, "think": False,
        "options": {"num_predict": 1, "temperature": 0, "seed": 0},
        "logprobs": True, "top_logprobs": 20,
    }).encode()
    request = urllib.request.Request(OLLAMA_URL + "/api/generate", body, {"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=TIMEOUT_SEC) as r:
        data = json.loads(r.read())
    if "error" in data:
        raise RuntimeError(data["error"])
    top = data["logprobs"][0]
    result = {top["token"].strip(): top["logprob"]}
    for alt in top.get("top_logprobs") or []:
        result.setdefault(alt["token"].strip(), alt["logprob"])
    return result


# 試しに 1 回聞いてみる（段階0 の見本と同じ形）
test = first_token_logprobs("Answer Yes or No.\nIs CFTR the gene mutated in cystic fibrosis?\nAnswer:")
print({k: round(v, 3) for k, v in list(test.items())[:5]})

## 8. 道具 2 — Bradley-Terry で「強さ」を出す

選択式の出題は、1 問ごとに「並んだ遺伝子それぞれが選ばれた確率」が得られます。
これを全部まとめて、**遺伝子ごとの強さ**を 1 つの数にします（Bradley-Terry / Luce のモデル）。

- 強い相手ばかりの群で勝った遺伝子は、弱い相手ばかりの群で勝った遺伝子より高く評価されます
  （平均確率で比べると、弱い群に当たり続けた遺伝子が得をします）。
- 「その他」「どちらも関係ない」も仮想の遺伝子 `NONE` として一緒に強さを出します。
  **NONE より弱い遺伝子は、「関係ない」に負けることが多かった遺伝子**です。
- 計算は MM 法の繰り返し（最尤推定）です。`ALPHA` は、強さ 1 の仮想の相手との引き分けを足す補正です。

返り値は `{遺伝子: log 強さ}` です。見やすさのため、表示では `elo = 1500 + 400 × log 強さ / log 10` に直します。

In [ ]:
def bradley_terry(matches, iterations=500, tolerance=1e-9):
    wins, groups_of = defaultdict(float), defaultdict(list)
    for group, probs in matches:                  # 勝ち数は確率の合計（小数で数える）
        for g in group:
            wins[g] += probs.get(g, 0.0)
            groups_of[g].append(group)
    strength = {g: 1.0 for g in wins}
    for _ in range(iterations):
        group_sum, new = {}, {}
        for g in strength:
            denom = 0.0
            for group in groups_of[g]:
                s = group_sum.get(id(group))
                if s is None:
                    s = group_sum[id(group)] = sum(strength[x] for x in group)
                denom += 1.0 / s
            denom += 2 * ALPHA / (strength[g] + 1.0)
            new[g] = (wins[g] + ALPHA) / denom
        scale = math.exp(statistics.mean(math.log(v) for v in new.values()))   # 全体の大きさをそろえる
        new = {g: v / scale for g, v in new.items()}
        change = max(abs(new[g] - strength[g]) for g in strength)
        strength = new
        if change < tolerance:
            break
    return {g: math.log(v) for g, v in strength.items()}


to_elo = lambda log_strength: 1500 + 400 * log_strength / math.log(10)
LETTERS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

## 9. 段階0 — 全候補を 1 遺伝子ずつ Yes / No で聞く

プロンプトは次の形です（見本 4 問のあとに、本番の 1 問）。

```
Answer Yes or No.

Disease: Cystic fibrosis
Symptoms to treat:
- Thick, sticky mucus that clogs the airways (lungs)
- Repeated lung infections (lungs)
Could changing this gene's activity, in either direction, potentially improve the disease or any of these symptoms of the disease?
Gene: CFTR
Answer: Yes
  （APOE → No、DNASE1 → Yes、HBB → No の見本が続く）

Disease: <対象の疾患>
Symptoms to treat:
- <症状 1（部位）>
- ...
Would changing ... ?
Gene: <記号>
Answer:
```

- **遺伝子記号をいちばん最後に置きます。** それより前は全遺伝子で共通なので、Ollama が計算済みの部分を使い回し、1 遺伝子 0.3 秒ほどで済みます。
- 点数は **log P(Yes) − log P(No)** です。正なら Yes 寄り、負なら No 寄り。
- 点数の高い順に `STAGE0_KEEP` 個を段階0.5 へ送ります。ここは「正解を落とさない」ための緩いふるいで、残した中の順番は問いません。
- 分かっている弱点: 上位は +25〜+30 に大勢が並び、差がつきません（飽和）。蛋白質名を足すと飽和は緩みますが、正解の順位はかえって下がりました。
- 結果は `stage0.tsv` に保存します。あれば読み込むだけです。

In [ ]:
disease_block = f"Disease: {DISEASE}\nSymptoms to treat:\n" + "".join(f"- {s}\n" for s in DISEASE_SYMPTOMS)

# 見本 4 問 ＋ 本番の問いの「Gene: 」まで（全遺伝子で共通の部分）
binary_prefix = "Answer Yes or No.\n\n"
for gene, answer in BINARY_EXAMPLES:
    binary_prefix += ("Disease: Cystic fibrosis\nSymptoms to treat:\n"
                      + "".join(f"- {s}\n" for s in FEWSHOT_SYMPTOMS["Cystic fibrosis"])
                      + f"{BINARY_QUESTION}\nGene: {gene}\nAnswer: {answer}\n\n")
binary_prefix += disease_block + f"{BINARY_QUESTION}\nGene: "

stage0_file = os.path.join(RUN_DIR, "stage0.tsv")
if os.path.exists(stage0_file):
    STAGE0_SCORE = {}
    with open(stage0_file) as f:
        next(f)
        for line in f:
            rank, gene, score = line.rstrip("\n").split("\t")
            STAGE0_SCORE[gene] = float(score)
    print(f"段階0 は保存済みを読み込みました（{len(STAGE0_SCORE)} 遺伝子）")
else:
    STAGE0_SCORE = {}
    start = time.time()
    for i, gene in enumerate(GENES, 1):
        logprobs = first_token_logprobs(binary_prefix + gene + "\nAnswer:")
        STAGE0_SCORE[gene] = logprobs.get("Yes", -99.0) - logprobs.get("No", -99.0)   # 上位 20 に無い側は -99
        if i == 10 or i % 500 == 0 or i == len(GENES):      # 最初の 10 件で全体の見積もりを出す（大きいモデル向け）
            elapsed = time.time() - start
            print(f"  {i}/{len(GENES)}  経過 {elapsed/60:.1f} 分  残り {(len(GENES)-i)*elapsed/i/60:.1f} 分")
    ranked = sorted(STAGE0_SCORE, key=lambda g: -STAGE0_SCORE[g])
    with open(stage0_file, "w") as f:
        f.write("rank\tgene\tscore\n")
        for rank, gene in enumerate(ranked, 1):
            f.write(f"{rank}\t{gene}\t{STAGE0_SCORE[gene]:.4f}\n")

STAGE0_RANK = {g: i for i, g in enumerate(sorted(STAGE0_SCORE, key=lambda g: -STAGE0_SCORE[g]), 1)}
STAGE0_PASS = sorted(STAGE0_SCORE, key=lambda g: -STAGE0_SCORE[g])[:STAGE0_KEEP]

print(f"\n段階0: {len(GENES)} → {len(STAGE0_PASS)}   Yes 側 {sum(v > 0 for v in STAGE0_SCORE.values())} 遺伝子")
print("上位 15:", ", ".join(f"{g}({STAGE0_SCORE[g]:+.1f})" for g in STAGE0_PASS[:15]))
for g in KNOWN_ANSWERS:
    if g in STAGE0_SCORE:
        print(f"  {g:<10} {STAGE0_RANK[g]:>6} 位  {STAGE0_SCORE[g]:+6.1f}  {'通過' if g in STAGE0_PASS else '⚠ 脱落'}")

## 10. 段階0.5 と 段階1 — 5 択＋「その他」で並べる

同じ処理を 2 回まわします（`CHOICE_ROUNDS` の 2 行）。1 回目（段階0.5）は粗く、2 回目（段階1）は詳しく並べます。

1 回分の手順:

1. **群を作る。** 候補をシャッフルした山札を「登場回数」の枚数だけつなげ、5 個ずつ区切ります。
   こうすると**全員がちょうど同じ回数**出題されます。同じ群に同じ遺伝子が 2 回入ったら、後ろの群と入れ替えます。
   （以前の勝ち残り方式では、早く落ちた遺伝子が 5 回、最後まで残った遺伝子が 35 回と、登場回数が 7 倍違いました。）
2. **出題する。** 選択肢の並びは問題ごとにシャッフルし、最後に「その他」を足します。
   モデルが答えの 1 文字目に `A`〜`F` のどれを出すかの確率を読み、選択肢ごとの確率に直します。
3. **Bradley-Terry で強さを出し**、上位 `残す数` 個を次へ送ります。表示する順位と次へ進む遺伝子は同じ基準です。

乱数の種を固定しているので、同じ設定なら毎回同じ群・同じ並びになります。
各回の出題は `stage0_5_matches.json` / `stage1_matches.json`、順位は `..._ranking.tsv` に保存します。

**注意:** 選択式は「5 つの中でいちばん」を選ばせるので、二値（段階0）より症状の書き方に強く左右されます。

In [ ]:
pool = list(STAGE0_PASS)
CHOICE_MATCHES = []          # 段階0.5・1 の全出題（最後の順位づけでも使う）
ROUND_RANK = {}              # 回ごとの順位（表示用）

# 選択式の見本（2 問）。本番と同じ形にする
choice_shots = ""
for disease, options, answer in CHOICE_EXAMPLES:
    lines = [f"{L}. {OPTION_TEXT[g]}" for L, g in zip(LETTERS, options)] + [f"{LETTERS[len(options)]}. {CHOICE_EXIT}"]
    choice_shots += (f"Disease: {disease}\nSymptoms to treat:\n" + "".join(f"- {s}\n" for s in FEWSHOT_SYMPTOMS[disease])
                     + CHOICE_QUESTION + "\n" + "\n".join(lines) + f"\nAnswer: {LETTERS[options.index(answer)]}\n\n")

for round_name, keep, appearances, seed in CHOICE_ROUNDS:
    keep = min(keep, len(pool))
    size = min(GROUP_SIZE, len(pool))
    file_key = "stage" + round_name.replace("段階", "").replace(".", "_")      # stage0_5 / stage1
    matches_file = os.path.join(RUN_DIR, f"{file_key}_matches.json")
    print(f"\n===== {round_name}: {len(pool)} 遺伝子 × {appearances} 回 ÷ {size} 個 = {-(-len(pool)*appearances//size)} 問")

    if os.path.exists(matches_file):
        with open(matches_file) as f:
            matches = [(tuple(m["group"]), m["probs"]) for m in json.load(f)]
        print(f"  保存済みを読み込みました（{len(matches)} 問）")
    else:
        # ----- 1. 群を作る（全員ちょうど appearances 回）-----
        rng = random.Random(seed)
        sequence = []
        for _ in range(appearances):
            deck = list(pool)
            rng.shuffle(deck)
            sequence += deck
        for i in range(len(sequence)):                  # 同じ群に同じ遺伝子が入ったら、後ろの群と入れ替える
            start = i - i % size
            if sequence[i] in sequence[start:i]:
                for j in range(start + size, len(sequence)):
                    j_start = j - j % size
                    if sequence[j] not in sequence[start:start + size] and sequence[i] not in sequence[j_start:j_start + size]:
                        sequence[i], sequence[j] = sequence[j], sequence[i]
                        break
        groups = [sequence[i:i + size] for i in range(0, len(sequence), size)]

        # ----- 2. 出題する -----
        random.seed(seed)                               # 選択肢の並べ替えも固定する
        matches, exits, started = [], 0, time.time()
        for n, group in enumerate(groups, 1):
            options = list(group)
            random.shuffle(options)
            letter_of = {L: g for L, g in zip(LETTERS, options + [NONE])}
            lines = [f"{L}. {OPTION_TEXT[g]}" for L, g in zip(LETTERS, options)] + [f"{LETTERS[len(options)]}. {CHOICE_EXIT}"]
            prompt = ("Answer with a single letter only.\n\n" + choice_shots + disease_block
                      + CHOICE_QUESTION + "\n" + "\n".join(lines) + "\nAnswer:")
            logprobs = first_token_logprobs(prompt)
            seen = {L: logprobs[L] for L in letter_of if L in logprobs}
            if not seen:
                print(f"  問 {n}: 選択肢の文字が返りませんでした（飛ばします）")
                continue
            floor = min(seen.values()) - 10.0               # 上位 20 に出なかった文字は、いちばん低い値よりさらに下に置く
            top = max(seen.values())
            weights = {L: math.exp(seen.get(L, floor) - top) for L in letter_of}
            total = sum(weights.values())
            probs = {letter_of[L]: w / total for L, w in weights.items()}   # 選択肢ごとの確率（合計 1）
            exits += max(probs, key=probs.get) == NONE
            matches.append((tuple(group) + (NONE,), probs))
            if n == 10 or n % 200 == 0 or n == len(groups):
                elapsed = time.time() - started
                print(f"  {n}/{len(groups)} 問  経過 {elapsed/60:.1f} 分  残り {(len(groups)-n)*elapsed/n/60:.1f} 分")
        print(f"  「その他」が 1 位になった問: {exits}/{len(matches)}")
        with open(matches_file, "w") as f:
            json.dump([{"group": list(g), "probs": {k: round(v, 6) for k, v in p.items()}} for g, p in matches], f)

    # ----- 3. 強さを出して、上位 keep 個を残す -----
    strength = bradley_terry(matches)
    ranked = sorted(pool, key=lambda g: -strength.get(g, -99))
    ROUND_RANK[round_name] = {g: i for i, g in enumerate(ranked, 1)}
    with open(os.path.join(RUN_DIR, f"{file_key}_ranking.tsv"), "w") as f:
        f.write("rank\tgene\telo\tabove_none\tstage0_rank\n")
        for i, g in enumerate(ranked, 1):
            f.write(f"{i}\t{g}\t{to_elo(strength[g]):.1f}\t{strength[g] > strength[NONE]}\t{STAGE0_RANK[g]}\n")
    print(f"  「その他」の線（NONE）: elo {to_elo(strength[NONE]):.0f}  → 線より上 {sum(strength[g] > strength[NONE] for g in pool)} 遺伝子")
    print("  上位 15:", ", ".join(f"{g}({to_elo(strength[g]):.0f})" for g in ranked[:15]))
    for g in KNOWN_ANSWERS:
        if g in ROUND_RANK[round_name]:
            r = ROUND_RANK[round_name][g]
            print(f"  {g:<10} {r:>5} 位  {'通過' if r <= keep else '脱落'}")

    CHOICE_MATCHES += matches
    pool = ranked[:keep]

FINALISTS = pool
print(f"\n段階2 へ: {len(FINALISTS)} 遺伝子")

## 11. 段階2 — 残った遺伝子の総当たり

残った遺伝子のすべての組を、2 つ＋「どちらも関係ない」の 3 択で聞きます。

- **A/B を入れ替えて 2 回聞きます**（`BOTH_WAYS`）。モデルには「先に書いた方を選びやすい」癖があるので、両方向で打ち消します。
- 聞く回数は `n × (n − 1)` 回です。100 遺伝子なら 9,900 回、1 回 0.6〜0.9 秒で約 2 時間かかります。
- **1 回ごとに `stage2_matches.jsonl` に追記します。** 途中で止めても、`RESUME_DIR` を指定して実行し直せば続きから始まります。
- 症状も仕組みも書かずに疾患名だけで聞くと、この形式はほぼ「いつも A を選ぶ」だけになりました。
- 症状の形では、有力な遺伝子どうし（DRD2 と C4A など）の対戦は、先に書いた方が勝つことが多くなりました（統合失調症で、今の質問文では 21 組中 17 組、「変える可能性があるか」では 18 組、「良くなるか」では 12 組、メカニズムの形では 0 組）。
  有力な遺伝子どうしの優劣は、段階2 ではほとんど決まらず、段階0.5・1 の結果が効きます。
  入れ替えた 2 回を合わせると引き分けになるので、`BOTH_WAYS` は切らないでください。

In [ ]:
# 1 対 1 の見本（2 問）
pair_shots = ""
for disease, options, answer in PAIR_EXAMPLES:
    lines = [f"{L}. {OPTION_TEXT[g]}" for L, g in zip(LETTERS, options)] + [f"C. {PAIR_EXIT}"]
    pair_shots += (f"Disease: {disease}\nSymptoms to treat:\n" + "".join(f"- {s}\n" for s in FEWSHOT_SYMPTOMS[disease])
                   + CHOICE_QUESTION + "\n" + "\n".join(lines) + f"\nAnswer: {LETTERS[options.index(answer)]}\n\n")

# 聞く組の一覧（A/B の順番つき）
orders = []
for i, a in enumerate(FINALISTS):
    for b in FINALISTS[i + 1:]:
        orders.append((a, b))
        if BOTH_WAYS:
            orders.append((b, a))

# 保存済みの対戦を読み込む（途中から再開するため）
pair_file = os.path.join(RUN_DIR, "stage2_matches.jsonl")
PAIR_MATCHES, done = [], set()
if os.path.exists(pair_file):
    with open(pair_file) as f:
        for line in f:
            m = json.loads(line)
            done.add(tuple(m["order"]))
            PAIR_MATCHES.append((tuple(m["group"]), m["probs"]))
todo = [o for o in orders if o not in done]
print(f"総当たり: {len(FINALISTS)} 遺伝子 → {len(orders)} 回（済み {len(done)}、残り {len(todo)}）")

started = time.time()
with open(pair_file, "a") as out:
    for n, (a, b) in enumerate(todo, 1):
        prompt = ("Answer with a single letter only.\n\n" + pair_shots + disease_block + CHOICE_QUESTION + "\n"
                  + f"A. {OPTION_TEXT[a]}\nB. {OPTION_TEXT[b]}\nC. {PAIR_EXIT}\nAnswer:")
        logprobs = first_token_logprobs(prompt)
        letter_of = {"A": a, "B": b, "C": NONE}
        seen = {L: logprobs[L] for L in letter_of if L in logprobs}
        if not seen:
            continue                                        # 保存しないので、再開時に聞き直される
        floor, top = min(seen.values()) - 10.0, max(seen.values())
        weights = {L: math.exp(seen.get(L, floor) - top) for L in letter_of}
        total = sum(weights.values())
        probs = {letter_of[L]: round(w / total, 6) for L, w in weights.items()}
        PAIR_MATCHES.append(((a, b, NONE), probs))
        out.write(json.dumps({"order": [a, b], "group": [a, b, NONE], "probs": probs}) + "\n")
        out.flush()
        if n == 10 or n % 500 == 0 or n == len(todo):
            elapsed = time.time() - started
            print(f"  {n}/{len(todo)}  経過 {elapsed/60:.1f} 分  残り {(len(todo)-n)*elapsed/n/60:.1f} 分")

first_wins = sum(1 for g, p in PAIR_MATCHES if max(p, key=p.get) == g[0])
print(f"A 側が選ばれた割合: {first_wins}/{len(PAIR_MATCHES)}（半分前後なら位置の偏りは小さい）")

## 12. 最終順位

段階0.5・1 の選択式と段階2 の総当たりを**まとめて 1 回の Bradley-Terry に入れて**、段階2 に進んだ遺伝子を並べます。
どちらも「群と、選択肢ごとの確率」という同じ形なので、そのまま合わせられます。

表の見方:

- `elo`: 強さ。差 400 で「10 倍選ばれやすい」。
- `関係あり`: 仮想の遺伝子 `NONE`（「その他」「どちらも関係ない」）より強いかどうか。遺伝子数が多いと、ほぼ全員が「関係あり」になります。
- `段階0 / 段階0.5 / 段階1`: 各段階での順位。

結果は `final_ranking.tsv` に保存します。

In [ ]:
final_strength = bradley_terry(CHOICE_MATCHES + PAIR_MATCHES)
FINAL = sorted(FINALISTS, key=lambda g: -final_strength[g])
none_line = final_strength[NONE]

with open(os.path.join(RUN_DIR, "final_ranking.tsv"), "w") as f:
    f.write("rank\tgene\telo\trelevant\tstage0_rank\t" + "".join(f"{name}_rank\t" for name in ROUND_RANK) + "protein\n")
    for i, g in enumerate(FINAL, 1):
        f.write(f"{i}\t{g}\t{to_elo(final_strength[g]):.1f}\t{final_strength[g] > none_line}\t{STAGE0_RANK[g]}\t"
                + "".join(f"{ROUND_RANK[name][g]}\t" for name in ROUND_RANK) + f"{PROTEIN.get(g, '')}\n")

print(f"「関係ない」の線（NONE）: elo {to_elo(none_line):.0f}  → 関係あり {sum(final_strength[g] > none_line for g in FINAL)} / {len(FINAL)}\n")
print(f"{'順位':>4}  {'遺伝子':<10} {'elo':>6}  {'関係あり':<6} {'段階0':>6}" + "".join(f" {name:>7}" for name in ROUND_RANK) + "  蛋白質名")
for i, g in enumerate(FINAL, 1):
    print(f"{i:>4}  {g:<10} {to_elo(final_strength[g]):>6.0f}  {'○' if final_strength[g] > none_line else '×':<6} {STAGE0_RANK[g]:>6}"
          + "".join(f" {ROUND_RANK[name][g]:>7}" for name in ROUND_RANK) + f"  {PROTEIN.get(g, '')[:50]}")

print("\n答え合わせ:")
for g in KNOWN_ANSWERS:
    if g in FINAL:
        print(f"  {g:<10} 最終 {FINAL.index(g) + 1} 位 / {len(FINAL)}")
    elif any(g in ROUND_RANK[name] for name in ROUND_RANK):
        last = [name for name in ROUND_RANK if g in ROUND_RANK[name]][-1]      # 最後に出題された回で落ちた
        print(f"  {g:<10} {last} で脱落（{ROUND_RANK[last][g]} 位）")
    elif g in STAGE0_RANK:
        print(f"  {g:<10} 段階0 で脱落（{STAGE0_RANK[g]} 位）")
print("\n保存先:", RUN_DIR, sorted(os.listdir(RUN_DIR)))

## 13. 結果の読み方

- **上位は「この疾患と書いた症状に関わる遺伝子」です。** 別の症状（たとえば陽性症状の代わりに陰性症状・認知機能）で回すと、
  上位は入れ替わります。大事な判断には、症状の組を変えて 2 通り以上回し、どちらでも上位に来る遺伝子を見てください。
- **今ある薬の標的が上に来やすい**ことがあります。「症状が良くなるか」と聞いていたときは、症状を抑える薬の標的（統合失調症なら HTR2A、DRD2）が有利でした。
  「疾患または症状を変える可能性があるか」では、原因に近い GRIN2A が戻りましたが（選択式 0.34 → 0.61）、HTR2A は 0.90 のままでした。
  今の「疾患またはその症状が良くなる可能性があるか」では GRIN2A 0.53、HTR2A 0.92 で、薬の標的への寄りは残ります。
- **下位や脱落は「関係ない」の証拠ではありません。** モデルが知らないつながり（例: 承認薬の標的 CHRM4 と統合失調症）は拾えません。
  既知の標的の確認には使えますが、新しい標的を探す一次スクリーニングとしては過信しないでください。
- **同じ系統の遺伝子はまとめて上位に来やすい**です（グルタミン酸受容体の各サブユニットなど）。系統内の区別は苦手です。
- **温度 0 で、群と並びも乱数の種で固定**しているので、同じ設定なら同じ結果が返ります。再現性を確かめるには `CHOICE_ROUNDS` の種を変えてください。